In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import skewness

# Create Spark session
spark = SparkSession.builder.appName("SkewnessExample").getOrCreate()

# Sample data - you can replace this with your own dataset
data = [(1, 10), (2, 20), (3, 30), (4, 40), (5, 50), (6, 1000)]

# Create DataFrame with columns "id" and "value"
df = spark.createDataFrame(data, ["id", "value"])

# Show the data
df.show()

# Calculate skewness on the "value" column
skewness_df = df.select(skewness("value").alias("skewness_value"))

# Show skewness result
skewness_df.show()

# Stop Spark session
spark.stop()


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import skewness, kurtosis, mean, stddev, col

# Create Spark session
spark = SparkSession.builder.appName("ComplexStatsExample").getOrCreate()

# Load dataset (replace with your own path)
df = spark.read.csv("path/to/your/data.csv", header=True, inferSchema=True)

# Show schema and a sample
df.printSchema()
df.show(5)

# Assume dataset has columns: "category", "value"

# 1. Data Cleaning: Filter rows with nulls in 'category' or 'value'
df_clean = df.filter(col("category").isNotNull() & col("value").isNotNull())

# 2. Remove outliers: keep only values within 3 stddev of mean
stats = df_clean.select(
    mean("value").alias("mean_value"),
    stddev("value").alias("stddev_value")
).collect()[0]

mean_val = stats['mean_value']
stddev_val = stats['stddev_value']

lower_bound = mean_val - 3 * stddev_val
upper_bound = mean_val + 3 * stddev_val

df_filtered = df_clean.filter((col("value") >= lower_bound) & (col("value") <= upper_bound))

# 3. Calculate statistics per category
stats_per_category = df_filtered.groupBy("category").agg(
    mean("value").alias("mean"),
    stddev("value").alias("stddev"),
    skewness("value").alias("skewness"),
    kurtosis("value").alias("kurtosis")
)

# Show final statistics
stats_per_category.show(truncate=False)

# 4. Save the stats to a CSV file
stats_per_category.coalesce(1).write.mode("overwrite").csv("output/stats_per_category")

# Stop Spark session
spark.stop()
